In [1]:
from pathlib import Path
import time

import numpy as np
import onnxruntime as ort
import torch
from torchvision import datasets, transforms

from devanagari_model import DevanagariCNN

TEST_DIR = Path("../data/DevanagariHandwrittenCharacterDataset/Test")
WEIGHTS_PATH = Path("hindi_cnn_weights_pytorch.pt")
ONNX_PATH = Path("hindi_cnn.onnx")
ONNX_INT8_PATH = Path("hindi_cnn_int8.onnx")

NUM_RUNS = 200

In [ ]:
transform = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
    ]
)

dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

image, _ = dataset[0]

image = image.unsqueeze(0)

In [ ]:
model = DevanagariCNN()

state_dict = torch.load(
    WEIGHTS_PATH,
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(state_dict)
model.eval()

onnx_session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

onnx_int8_session = ort.InferenceSession(
    str(ONNX_INT8_PATH),
    providers=["CPUExecutionProvider"],
)

input_name = onnx_session.get_inputs()[0].name

In [ ]:
def benchmark_pytorch(model, image):

    with torch.no_grad():

        # warmup
        for _ in range(20):
            model(image)

        times = []

        for _ in range(NUM_RUNS):
            start = time.perf_counter()

            model(image)

            end = time.perf_counter()

            times.append((end - start) * 1000)

    return np.array(times)


def benchmark_onnx(session, image):

    image_np = image.numpy()

    input_name = session.get_inputs()[0].name

    # warmup
    for _ in range(20):
        session.run(None, {input_name: image_np})

    times = []

    for _ in range(NUM_RUNS):

        start = time.perf_counter()

        session.run(None, {input_name: image_np})

        end = time.perf_counter()

        times.append((end - start) * 1000)

    return np.array(times)


def summarize(name, times, model_path):

    mean = np.mean(times)
    p95 = np.percentile(times, 95)
    size = Path(model_path).stat().st_size / (1024 * 1024)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Mean latency : {mean:.3f} ms")
    print(f"P95 latency  : {p95:.3f} ms")
    print(f"Model size   : {size:.3f} MB")

    return mean, p95, size

In [ ]:
print("Running benchmarks...\n")

results = [
    (
        "PyTorch",
        "FP32",
        summarize("PyTorch FP32", benchmark_pytorch(model, image), WEIGHTS_PATH),
    ),
    (
        "ONNX Runtime",
        "FP32",
        summarize("ONNX FP32", benchmark_onnx(onnx_session, image), ONNX_PATH),
    ),
    (
        "ONNX Runtime",
        "INT8",
        summarize(
            "ONNX INT8", benchmark_onnx(onnx_int8_session, image), ONNX_INT8_PATH
        ),
    ),
]


print("\n Benchmark Table\n")

print(
    f"{'Runtime':<18}"
    f"{'Precision':<12}"
    f"{'Mean Latency':>15}"
    f"{'P95 Latency':>15}"
    f"{'Size':>12}"
)

print("-" * 72)

for runtime, precision, (mean, p95, size) in results:
    print(
        f"{runtime:<18}"
        f"{precision:<12}"
        f"{mean:>12.3f} ms"
        f"{p95:>12.3f} ms"
        f"{size:>9.3f} MB"
    )

Running benchmarks...


PyTorch FP32
------------
Mean latency : 2.128 ms
P95 latency  : 3.855 ms
Model size   : 2.305 MB

ONNX FP32
---------
Mean latency : 0.520 ms
P95 latency  : 0.861 ms
Model size   : 2.299 MB

ONNX INT8
---------
Mean latency : 0.612 ms
P95 latency  : 1.310 ms
Model size   : 0.771 MB

 Benchmark Table

Runtime           Precision      Mean Latency    P95 Latency        Size
------------------------------------------------------------------------
PyTorch           FP32               2.128 ms       3.855 ms    2.305 MB
ONNX Runtime      FP32               0.520 ms       0.861 ms    2.299 MB
ONNX Runtime      INT8               0.612 ms       1.310 ms    0.771 MB
